In [1]:
# Loading Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from xgboost import XGBClassifier
from sklearn.model_selection import TimeSeriesSplit

sns.set_theme(style="whitegrid")

df_raw = pd.read_csv('SP500_daily_update - kaggle.csv', header=[0, 1], index_col=0)
df_info = pd.read_csv('sp500live - info.csv')

In [2]:
#reshaping data
df_long = df_raw.stack(level=1).reset_index()
df_long.columns = ['Date', 'Ticker', 'Close', 'High', 'Low', 'Open', 'Volume']

df_info = df_info[['Ticker', 'Company', 'Sector', 'Industry']]
df_info['Ticker'] = df_info['Ticker'].str.replace('.', '-', regex=False)

#merge
df = pd.merge(df_long, df_info, on='Ticker', how='left')

# Final Cleaning
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values(['Ticker', 'Date']).reset_index(drop=True)

In [3]:
# Returns
df['Return_1'] = df.groupby('Ticker')['Close'].pct_change()
df['Return_5'] = df.groupby('Ticker')['Close'].pct_change(5)

# Moving averages
df['MA_5'] = df.groupby('Ticker')['Close'].transform(lambda x: x.rolling(5).mean())
df['MA_10'] = df.groupby('Ticker')['Close'].transform(lambda x: x.rolling(10).mean())

# Momentum
df['Momentum'] = df['Close'] - df['MA_5']

# Volatility
df['Volatility'] = df.groupby('Ticker')['Return_1'].transform(lambda x: x.rolling(5).std())

In [4]:
# RSI
def compute_rsi(series, window=14):
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.rolling(window).mean()
    avg_loss = loss.rolling(window).mean()
    rs = avg_gain / avg_loss
    return 100 - (100 / (1 + rs))

df['RSI'] = df.groupby('Ticker')['Close'].transform(compute_rsi)

# MA ratio
df['MA_ratio'] = df['Close'] / df['MA_10']

# Volume change
df['Volume_Change'] = df.groupby('Ticker')['Volume'].pct_change()


In [5]:
# Lag features (memory)
df['Return_1_lag1'] = df.groupby('Ticker')['Return_1'].shift(1)
df['Return_1_lag2'] = df.groupby('Ticker')['Return_1'].shift(2)

# Interaction feature
df['Momentum_RSI'] = df['Momentum'] * df['RSI']

In [6]:
#target variable
df['Target'] = (df.groupby('Ticker')['Close'].shift(-1) > df['Close']).astype(int)

#drop missing values
df_model = df.dropna().copy()

In [7]:
# Keep only 50 stocks
top_tickers = df_model['Ticker'].unique()[:50]
df_model = df_model[df_model['Ticker'].isin(top_tickers)]

# Use recent data only
df_model = df_model[df_model['Date'] >= '2023-01-01']

In [ ]:
features = [
    'Return_1', 'Return_5',
    'MA_5', 'MA_10',
    'Momentum', 'Volatility',
    'RSI', 'MA_ratio',
    'Volume_Change',
    'Return_1_lag1', 'Return_1_lag2',
    'Momentum_RSI'
]

#train per stock
results = []

for ticker in df_model['Ticker'].unique():
    
    df_ticker = df_model[df_model['Ticker'] == ticker].copy()
    
    if len(df_ticker) < 250:
        continue
    
    X = df_ticker[features]
    y = df_ticker['Target']
    
    tscv = TimeSeriesSplit(n_splits=5)
    
    acc_scores = []
    
    for train_idx, test_idx in tscv.split(X):
        
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        
        # Handle imbalance
        scale_pos_weight = len(y_train[y_train == 0]) / len(y_train[y_train == 1])
        
        model = XGBClassifier(
            n_estimators=300,
            max_depth=3,
            learning_rate=0.05,
            subsample=0.7,
            colsample_bytree=0.7,
            reg_alpha=1,
            reg_lambda=1,
            scale_pos_weight=scale_pos_weight,
            random_state=42,
            eval_metric='logloss'
        )
        
        model.fit(
            X_train, y_train,
            eval_set=[(X_test, y_test)],
            verbose=False
        )
        
        y_prob = model.predict_proba(X_test)[:, 1]
        
        # Threshold optimization 
        best_thresh = 0.5
        best_f1 = 0
        
        for t in np.arange(0.45, 0.65, 0.01):
            y_pred_temp = (y_prob > t).astype(int)
            score = f1_score(y_test, y_pred_temp)
            
            if score > best_f1:
                best_f1 = score
                best_thresh = t
        
        y_pred = (y_prob > best_thresh).astype(int)
        
        acc = accuracy_score(y_test, y_pred)
        acc_scores.append(acc)
    
    results.append({
        'Ticker': ticker,
        'Avg_Accuracy': np.mean(acc_scores),
        'Std_Accuracy': np.std(acc_scores)
    })

In [9]:
results_df = pd.DataFrame(results)

print("\n=== TOP PERFORMING STOCKS ===")
print(results_df.sort_values('Avg_Accuracy', ascending=False).head(10))

print("\n=== OVERALL PERFORMANCE ===")
print("Average Accuracy:", results_df['Avg_Accuracy'].mean())
print("\nSummary:")
print(results_df.describe())


=== TOP PERFORMING STOCKS ===
   Ticker  Avg_Accuracy  Std_Accuracy
44    AVB      0.558462      0.033916
40   APTV      0.556923      0.016570
36    APD      0.546154      0.039822
45   AVGO      0.541538      0.040587
5    ACGL      0.535385      0.015839
27    AME      0.530769      0.017541
42   ARES      0.526154      0.026469
48   AXON      0.526154      0.040295
46    AVY      0.526154      0.044216
35    APA      0.524615      0.046256

=== OVERALL PERFORMANCE ===
Average Accuracy: 0.5108923076923076

Summary:
       Avg_Accuracy  Std_Accuracy
count     50.000000     50.000000
mean       0.510892      0.034371
std        0.018325      0.016234
min        0.470769      0.009730
25%        0.500000      0.025925
50%        0.510000      0.031227
75%        0.522308      0.043414
max        0.558462      0.085159
